# Pennsylvania Hospital Accessibility — Multi-Download Workflow


**Real-world question:** *Which Pennsylvania counties are well-served (versus poorly-served) by hospitals within a 10-mile drive band?*


## Install GAS Client SDK

In [ ]:
%pip install -q gas-client

## Imports

In [ ]:
from urllib.parse import urljoin
from IPython.display import HTML, Image, display
from gas_client import GasClient

## User Settings


In [ ]:
server_url = "http://127.0.0.1:4042"
openai_api_key = "YOUR_OPENAI_API_KEY"
timeout_seconds = 2400

default_credentials = {"OPENAI_API_KEY": openai_api_key}

client = GasClient(
    server_url,
    default_credentials=default_credentials,
    artifact_delivery="URL",
)

## Bind to the Three Agents

In [3]:
data_agent = client.agent("geospatial_data_retrieval_agent")
vector_agent = client.agent("vector_analysis_agent")
mapping_agent = client.agent("mapping_agent")

for agent in [data_agent, vector_agent, mapping_agent]:
    print(agent.agent_id, agent.status().get("status"))

geospatial_data_retrieval_agent available
vector_analysis_agent available
mapping_agent available


## Helper Functions

In [4]:
def absolute_url(url):
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def all_artifact_urls(task_result):
    """Return every artifact URL from a (possibly multi-dataset) task response, in order."""
    artifacts = task_result.get("outputs", {}).get("artifacts", []) or []
    urls = [absolute_url(a["url"]) for a in artifacts if a.get("url")]
    return urls


def first_artifact_url(task_result, preferred_extensions=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", []) or []
    preferred_extensions = preferred_extensions or []

    for extension in preferred_extensions:
        for artifact in artifacts:
            url = artifact.get("url")
            filename = artifact.get("filename") or artifact.get("name") or url or ""
            if url and str(filename).lower().endswith(extension.lower()):
                return absolute_url(url)

    for artifact in artifacts:
        if artifact.get("url"):
            return absolute_url(artifact["url"])

    raise RuntimeError("The task returned no artifact URL.")


def pick_url_by_keywords(urls, keywords):
    """Pick the first URL whose filename contains any of the given keywords."""
    for url in urls:
        lower = url.lower()
        if any(k.lower() in lower for k in keywords):
            return url
    return None


def display_visual_artifacts(task_result):
    artifacts = task_result.get("outputs", {}).get("artifacts", []) or []
    for artifact in artifacts:
        url = artifact.get("url")
        filename = artifact.get("filename") or artifact.get("name") or ""
        if not url:
            continue

        display_url = absolute_url(url)
        lower_name = str(filename).lower()
        if lower_name.endswith(".png"):
            display(Image(url=display_url))
        elif lower_name.endswith(".html"):
            display(HTML(f'<a href="{display_url}" target="_blank">Open HTML artifact: {filename}</a>'))
        else:
            display(HTML(f'<a href="{display_url}" target="_blank">Open artifact: {filename}</a>'))


def run_streaming_task(agent, instructions, *, input_datasets=None, title=None):
    if title:
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)

    final_result = None
    for event in agent.execute_task(
        instructions,
        mode="stream",
        input_datasets=input_datasets,
        artifact_delivery="URL",
        timeout=timeout_seconds,
    ):
        client.print_stream_event(event)
        if event.get("event") == "task_result":
            final_result = event.get("payload")

    if final_result is None:
        raise RuntimeError("The stream ended before returning a task_result event.")

    client.print_task_summary(final_result)
    return final_result

## Step 1: One Retrieval Call → Three Datasets


In [5]:
multi_retrieval_result = run_streaming_task(
    data_agent,
    (
        "Please download three Pennsylvania datasets:\n"
        "1) Pennsylvania county boundaries from the U.S. Census Bureau TIGER source. "
        "Return a clean GeoPackage with county geometry, GEOID, county name, and STATEFP fields. "
        "Restrict the output strictly to Pennsylvania (STATEFP = 42).\n"
        "2) Pennsylvania hospitals from OpenStreetMap (amenity=hospital and healthcare=hospital). "
        "Return a clean point GeoPackage covering the state of Pennsylvania, with at least name and "
        "the original OSM tags retained where available.\n"
        "3) Pennsylvania major highways from OpenStreetMap (highway=motorway and highway=trunk). "
        "Return a clean line GeoPackage covering the state of Pennsylvania with name and highway-type fields.\n\n"
        "All three datasets should use WGS84 (EPSG:4326). No source API key is required."
    ),
    title="Multi-Dataset Retrieval: PA Counties + Hospitals + Highways",
)


Multi-Dataset Retrieval: PA Counties + Hospitals + Highways
[21:03:58] stream_connected: Streaming connection established.
[21:03:58] Geospatial Data Retrieval Agent: I received your request.
[21:03:58] Geospatial Data Retrieval Agent: I need to inspect the request text and any provided dataset references before running the agent. I found 0 dataset reference(s).
[21:03:58] Geospatial Data Retrieval Agent: I found the required credentials and can start the model-backed workflow.
[21:03:58] task_accepted: Task accepted. Starting streaming execution.
[21:03:58] Geospatial Data Retrieval Agent: Next I will run the agent with the prepared inputs.
[21:03:59] Geospatial Data Retrieval Agent: I am checking whether the request asks for one dataset or several.
[21:04:00] Geospatial Data Retrieval Agent: The request was decomposed into 3 sub-request(s).
[21:04:00] Geospatial Data Retrieval Agent: I detected 3 datasets in your request. I will download each one as a separate sub-task and return al

### Inspect the Multi-Download Response


In [6]:
metrics = multi_retrieval_result.get("metrics", {})
print("sub_task_count       :", metrics.get("sub_task_count"))
print("successful_sub_tasks :", metrics.get("successful_sub_tasks"))
print("failed_sub_tasks     :", metrics.get("failed_sub_tasks"))
print("total artifacts      :", metrics.get("number_of_artifacts"))

downloaded_urls = all_artifact_urls(multi_retrieval_result)
for url in downloaded_urls:
    print("  -", url)

sub_task_count       : None
successful_sub_tasks : None
failed_sub_tasks     : None
total artifacts      : None
  - http://127.0.0.1:4042/agents/geospatial_data_retrieval_agent/data/geospatial_data_retrieval_agent-7639-cgko-5924.gpkg
  - http://127.0.0.1:4042/agents/geospatial_data_retrieval_agent/data/geospatial_data_retrieval_agent-6475-xnip-4041.gpkg
  - http://127.0.0.1:4042/agents/geospatial_data_retrieval_agent/data/geospatial_data_retrieval_agent-5838-uejr-3268.gpkg


In [ ]:
if len(downloaded_urls) < 3:
    raise RuntimeError(
        f"Expected 3 downloaded artifacts (counties, hospitals, highways) but got {len(downloaded_urls)}. "
        "Inspect the multi-retrieval task output above before proceeding.\n"
        f"Artifacts: {downloaded_urls}"
    )

counties_url, hospitals_url, highways_url = downloaded_urls[0], downloaded_urls[1], downloaded_urls[2]

print("counties :", counties_url)
print("hospitals:", hospitals_url)
print("highways :", highways_url)

sub_tasks = multi_retrieval_result.get("outputs", {}).get("sub_tasks", []) or []
if sub_tasks:
    print("\nSub-task breakdown (verify the request-to-artifact mapping):")
    for record in sub_tasks:
        req_preview = (record.get("sub_request") or "")[:90]
        print(
            f"  #{record.get('sub_request_index')} "
            f"({record.get('artifact_count')} artifact(s)): "
            f"{req_preview}..."
        )

counties : http://127.0.0.1:4042/agents/geospatial_data_retrieval_agent/data/geospatial_data_retrieval_agent-7639-cgko-5924.gpkg
hospitals: http://127.0.0.1:4042/agents/geospatial_data_retrieval_agent/data/geospatial_data_retrieval_agent-6475-xnip-4041.gpkg
highways : http://127.0.0.1:4042/agents/geospatial_data_retrieval_agent/data/geospatial_data_retrieval_agent-5838-uejr-3268.gpkg


## Step 2: Compute 10-Mile Hospital Coverage Per County


In [12]:
coverage_result = run_streaming_task(
    vector_agent,
    (
        "Compute hospital accessibility per Pennsylvania county using the provided layers. "
        "Inputs (in this order): (1) PA county polygons, (2) PA hospital points, (3) PA highways (not used in the analysis, ignore). "
        "Steps: "
        "a) Reproject counties and hospitals to EPSG:5070 (NAD83 Conus Albers, units = meters) for buffering and area math. "
        "b) Buffer each hospital point by 16093 meters (10 miles) and dissolve all buffers into one coverage polygon. "
        "c) Intersect the dissolved coverage with each county; compute covered_area_m2 per county (0 if no overlap). "
        "d) Compute county_area_m2 from the projected county geometry and a new numeric field "
        "   coverage_pct = covered_area_m2 / county_area_m2 * 100, clamped to [0, 100]. "
        "e) Reproject the result back to EPSG:4326. "
        "Return one GeoPackage of PA counties with original identifiers (GEOID, NAME) and the new "
        "coverage_pct field. Do not return the hospital or highway layers from this step."
    ),
    input_datasets=[counties_url, hospitals_url, highways_url],
    title="Compute 10-mile Hospital Coverage per PA County",
)

coverage_url = first_artifact_url(
    coverage_result, preferred_extensions=[".gpkg", ".geojson", ".json"]
)
coverage_url


Compute 10-mile Hospital Coverage per PA County
[21:15:51] stream_connected: Streaming connection established.
[21:15:51] Vector Analysis Agent: I received your request.
[21:15:51] Vector Analysis Agent: I need to inspect the request text and any provided dataset references before running the agent. I found 3 dataset reference(s).
[21:15:51] Vector Analysis Agent: I found the required credentials and can start the model-backed workflow.
[21:15:52] task_accepted: Task accepted. Starting streaming execution.
[21:15:52] Vector Analysis Agent: Next I will run the agent with the prepared inputs.
[21:15:52] Vector Analysis Agent: I will load the requested vector/tabular inputs, run code-driven analysis, and save a final dataset artifact from 3 dataset reference(s).
[21:15:52] Vector Analysis Agent: I detected a common vector operation and will first try a deterministic GeoPandas workflow.
[21:15:52] Vector Analysis Agent: Added geometry measurement field(s): area_sq_m, area_sq_km and saved 

'http://127.0.0.1:4042/agents/vector_analysis_agent/data/vector_analysis_agent-0953-hsbw-9392.gpkg'

## Step 3: Map the Result

The mapping agent renders the choropleth of `coverage_pct` with hospitals and major highways overlaid for context.

In [13]:
map_result = run_streaming_task(
    mapping_agent,
    (
        "Create a professional county-level choropleth map of Pennsylvania showing 10-mile hospital coverage. "
        "Inputs (in this order): (1) PA counties with a coverage_pct field, (2) PA hospital points, "
        "(3) PA major highways. "
        "Use coverage_pct for county fill color, a 5-class sequential green color ramp (light = low coverage, "
        "dark = high coverage), thin dark-gray county outlines, hospital points as small red dots on top, "
        "and highways as semi-transparent dark gray lines beneath the hospital points. "
        "Use the EPSG:5070 (Conus Albers) projection, a legend outside the map area listing the choropleth "
        "breaks and a small inset legend for hospitals and highways, the title "
        "'Pennsylvania Counties: % Area Within 10 Miles of a Hospital', and a small data-source caption "
        "crediting OpenStreetMap and U.S. Census TIGER."
    ),
    input_datasets=[coverage_url, hospitals_url, highways_url],
    title="Render the PA Hospital Coverage Map",
)

display_visual_artifacts(map_result)


Render the PA Hospital Coverage Map
[21:16:05] stream_connected: Streaming connection established.
[21:16:05] Mapping Agent: I received your request.
[21:16:05] Mapping Agent: I need to inspect the request text and any provided dataset references before running the agent. I found 3 dataset reference(s).
[21:16:06] Mapping Agent: I found the required credentials and can start the model-backed workflow.
[21:16:06] task_accepted: Task accepted. Starting streaming execution.
[21:16:06] Mapping Agent: Next I will run the agent with the prepared inputs.
[21:16:06] Mapping Agent: I will inspect the requested visualization and the 3 dataset reference(s), then choose whether a map or chart is the best way to answer it.
[21:16:06] Mapping Agent: I am drafting visualization code now. This is attempt 1; I will run the code and check whether it creates the requested output correctly.
[21:16:16] Mapping Agent: I'm still working. Long LLM calls, code execution, or geospatial file processing can take